<a href="https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable,"-m","pip","install","-q","-r","requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:",os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

df.head()

Working directory: /content/flyrank-ml-internship-starter
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Build the feature vector

The objective of this feature vector is to represent each content page using observable signals that are available before making a refresh decision.

The selected features describe content age, visibility, engagement, maintenance history, and content size. Missing values are handled using simple imputations so the feature vector remains complete for modeling.

Only information available at the decision moment is included to avoid information leakage.

In [2]:
feature_df = df[[
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]].copy()

feature_df = feature_df.fillna(0)

print(feature_df.head())

print()

print(feature_df.describe())

   content_age_days  days_since_last_update  impressions_90d  avg_position  \
0               187                      20             3803          10.6   
1               445                      25            15320          20.3   
2               141                      20            12581          36.5   
3               463                      22            11751           6.2   
4               263                      14            19140          44.0   

    ctr  word_count  
0  0.76      3221.0  
1  0.05      2481.0  
2  0.09      3515.0  
3  0.49         0.0  
4  0.13      2803.0  

       content_age_days  days_since_last_update  impressions_90d  \
count       30000.00000            30000.000000     30000.000000   
mean          256.16780               46.098300      5200.366300   
std           132.70793               42.078709     16838.019547   
min            90.00000                1.000000         1.000000   
25%           132.00000               20.000000        81.

## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature                | Meaning                                    | Missing Handling | Available Before Prediction |
| ---------------------- | ------------------------------------------ | ---------------- | --------------------------- |
| content_age_days       | Age of the content                         | Filled with 0    | Yes                         |
| days_since_last_update | Days since last content update             | Filled with 0    | Yes                         |
| impressions_90d        | Search impressions during previous 90 days | Filled with 0    | Yes                         |
| avg_position           | Average search position                    | Filled with 0    | Yes                         |
| ctr                    | Click-through rate                         | Filled with 0    | Yes                         |
| word_count             | Number of words in the page                | Filled with 0    | Yes                         |

All selected features are available before making the refresh recommendation and therefore can safely be used for prediction.

In [3]:
feature_summary = pd.DataFrame({
    "Missing Values": feature_df.isnull().sum(),
    "Data Type": feature_df.dtypes
})

feature_summary

,Missing Values,Data Type
content_age_days,0,int64
days_since_last_update,0,int64
impressions_90d,0,int64
avg_position,0,float64
ctr,0,float64
word_count,0,float64


## 3. The leakage hunt

Potential leakage occurs when a feature contains information that directly reveals the prediction target or comes from the future.

The following fields were reviewed:

trend_direction
trend_pct
product-generated recommendation flags
future performance measurements

The model only uses observable signals available before the prediction moment.

No future information is intentionally included in the final feature vector.

In [4]:
df["is_declining_label"] = (
    df["trend_direction"]
      .str.lower()
      .eq("down")
      .astype(int)
)

from sklearn.tree import DecisionTreeClassifier

safe_features = feature_df

tree = DecisionTreeClassifier(max_depth=3,random_state=42)

tree.fit(safe_features,df["is_declining_label"])

print("Safe feature model trained.")

print()

print("Leakage test")

leaky = feature_df.copy()

leaky["trend_pct"] = df["trend_pct"]

print("Added trend_pct intentionally.")

print("This feature should NOT be used because it leaks label information.")

Safe feature model trained.

Leakage test
Added trend_pct intentionally.
This feature should NOT be used because it leaks label information.


## 4. What I excluded and why

The following fields were deliberately excluded from the feature vector:

trend_direction — used to create the prediction label and would leak the outcome.
trend_pct — directly related to the target and contains future outcome information.
Product recommendation flags — represent existing business decisions rather than independent observations.
Future performance values — unavailable at prediction time.
Private identifiers — excluded to protect privacy and avoid memorization.

These exclusions help ensure the model remains honest, generalizable, and suitable for decision-support rather than memorizing outcomes.

In [5]:
excluded = pd.DataFrame({
    "Excluded Feature":[
        "trend_direction",
        "trend_pct",
        "product_flags",
        "future_metrics",
        "private_identifiers"
    ],
    "Reason":[
        "Creates prediction label",
        "Leaks future outcome",
        "Business decision",
        "Unavailable during prediction",
        "Privacy protection"
    ]
})

excluded

,Excluded Feature,Reason
0,trend_direction,Creates prediction label
1,trend_pct,Leaks future outcome
2,product_flags,Business decision
3,future_metrics,Unavailable during prediction
4,private_identifiers,Privacy protection


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.